# CatRaRE heavy-rainfall events — a guided tour

This notebook teaches the [`catrare`](../../reference/catrare/introduction.md) backend: how to fetch the **DWD CatRaRE** catalogue of radar-based heavy-rainfall **events** over Germany (2001-2025, derived from the RADKLIM radar climatology) and explore it. The data is public (`opendata.dwd.de`, CC-BY-4.0 / GeoNutzV, no credentials).

Each event is a space-time cluster of intense rainfall with attributes such as duration, area, maximum rainfall, and a severity index (`Eta`). The backend downloads a FileGDB, reprojects it off the DWD RADOLAN grid to EPSG:4326, and returns the events as vector features filtered by date and bounding box.

## The request axes

| Argument | Meaning | Values |
|---|---|---|
| `threshold` | which selection | `t5` (return period >= 5 yr), `w3` (severity-weighted) |
| `geometry_layer` | which layer | `zones` (event-footprint polygons), `points` (max-rainfall points) |
| `start` / `end` | date window | keep events overlapping it |
| `lat_lim` / `lon_lim` | region | a WGS84 bounding box |
| `geometry` | output shape | `True` -> `FeatureCollection`, `False` -> `DataFrame` |

## Setup

Imports and a single notebook-relative cache/output dir (`out/`, gitignored).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from earthlens.core import EarthLens

DATA = "out"

## Quickstart — the July 2021 flood over west Germany

The shortest end-to-end call: the **T5** heavy-rainfall events in July 2021 over the Ahr / North Rhine-Westphalia region. The first call downloads the ~16 MB T5 FileGDB and caches it under `out/`; later calls reuse it.

In [ ]:
ahr = EarthLens(
    "catrare",
    threshold="t5",
    start="2021-07-01",
    end="2021-07-31",
    lat_lim=[50.0, 51.5],
    lon_lim=[6.0, 8.0],
    path=DATA,
    cache_dir=DATA,
).download()

print(len(ahr), "events |", list(ahr.columns)[:6])

Each event is a footprint polygon; colour them by the severity index `Eta`:

In [ ]:
ax = ahr.plot(
    column="Eta",
    cmap="Reds",
    legend=True,
    legend_kwds={"label": "Severity index (Eta)", "shrink": 0.6},
    figsize=(7, 7),
    edgecolor="black",
    linewidth=0.3,
)
ax.set_title("CatRaRE T5 events - Ahr / NRW, July 2021")
ax.set_axis_off()
plt.show()

The mid-July cluster is the catastrophic 2021 flood: a dense band of high-severity events across the Ahr and Erft catchments.

## The full archive — events through time

Drop the date/bbox filter (and `geometry=False`) to pull the whole 2001-2025 T5 catalogue as a table. This reuses the cached FileGDB. Counting events per year shows the interannual variability of heavy rainfall.

In [ ]:
allt5 = EarthLens(
    "catrare",
    threshold="t5",
    geometry=False,
    path=DATA,
    cache_dir=DATA,
).download()

print(len(allt5), "T5 events, 2001-2025")

In [ ]:
per_year = pd.to_datetime(allt5["Date_START"]).dt.year.value_counts().sort_index()
ax = per_year.plot.bar(figsize=(11, 3.4), color="#2980b9")
ax.set_ylabel("T5 events")
ax.set_title("Heavy-rainfall (T5) events per year")
plt.tight_layout()
plt.show()

The severity index `Eta` is right-skewed — most events are moderate, a long tail is extreme:

In [ ]:
ax = allt5["Eta"].plot.hist(bins=40, figsize=(6, 3.4), color="#c0392b")
ax.set_xlabel("Severity index (Eta)")
ax.set_title("Distribution of T5 event severity")
plt.tight_layout()
plt.show()

## T5 vs W3 threshold

`threshold="t5"` selects events reaching a 5-year return period; `threshold="w3"` is a severity-weighted selection. They ship as separate FileGDBs (the W3 pull downloads its own).

In [ ]:
w3 = EarthLens(
    "catrare",
    threshold="w3",
    geometry=False,
    path=DATA,
    cache_dir=DATA,
).download()

pd.Series({"T5": len(allt5), "W3": len(w3)})

## Points instead of zones

`geometry_layer="points"` returns one maximum-rainfall **point** per event instead of the footprint polygon — handy for a density map or a join to station data.

In [ ]:
pts = EarthLens(
    "catrare",
    threshold="t5",
    geometry_layer="points",
    start="2021-07-01",
    end="2021-07-31",
    lat_lim=[50.0, 51.5],
    lon_lim=[6.0, 8.0],
    path=DATA,
    cache_dir=DATA,
).download()

set(pts.geometry.geom_type)

## Takeaway

- One call shape — `threshold` x `geometry_layer` x date window x bbox x `geometry` — covers the product.
- Events carry `Area`, `Duration`, `RRmax`, and the `Eta` severity index; footprints are polygons, `RRmaxPoints` are points.
- The geometry is reprojected off the DWD RADOLAN grid to EPSG:4326, so lat/lon boxes just work.
- CatRaRE is the **event companion** to the RADKLIM grids — use it for pluvial-flood and cloudburst analysis.

See the [CatRaRE reference](../../reference/catrare/introduction.md) for the full attribute list and licence.